# Od słów do attention: budujemy blok transformera

## Zadanie
Zbuduj w PyTorchu elementy modelu języka: tokenizację, embeddingi, kodowanie pozycji, skalowane attention, maskę przyczynową, multi-head attention i blok transformera. Sprawdzisz kształty tensorów oraz wpływ maski. To pierwsza część ścieżki; recenzje win są przykładem tekstu.

Wymagane: `torch`, `matplotlib`. Przykłady działają na CPU lub CUDA.

## 1. Korpus i tokenizacja
Pełny Wine Reviews dataset zwykle wymaga osobnego pobrania. Żeby notebook działał offline, korzystamy z małego, dydaktycznego korpusu. Nie jest statystycznie reprezentatywny; można go podmienić na lokalne CSV.

In [ ]:
import math,re,torch
from torch import nn
import matplotlib.pyplot as plt
torch.manual_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
reviews=["aromatic cherry and plum with a long finish","bright acidity balances ripe citrus fruit","dry wine with notes of pear and mineral","soft tannins frame dark berry and spice","fresh apple aroma with crisp acidity","rich oak vanilla and blackcurrant flavors","light bodied wine with floral notes","ripe peach and honey with a smooth finish","structured tannins and earthy cherry notes","zesty lemon aroma and clean mineral finish","bold dark fruit with pepper and toasted oak","delicate strawberry notes and lively acidity"]
def tokenize(s): return re.findall(r"[a-z]+|[^\w\s]",s.lower())
special=["<pad>","<unk>","<bos>","<eos>"]
vocab=special+sorted({t for s in reviews for t in tokenize(s)})
stoi={t:i for i,t in enumerate(vocab)}
ids=torch.tensor([[stoi.get(t,1) for t in tokenize(reviews[0])]],device=device)
embedding=nn.Embedding(len(vocab),16).to(device)
print("device",device,"vocab",len(vocab),"IDs",ids.shape,"embedding",embedding(ids).shape)

## 2. Scaled dot-product attention
Dla zapytań $Q$, kluczy $K$ i wartości $V$:

$$Attention(Q,K,V)=softmax(QK^T/√d_k+M)V.$$

$QK^T$ mierzy zgodność zapytania z kluczem. Dzielenie przez $√d_k$ stabilizuje wariancję logitów. Maska $M$ dodaje bardzo małą wartość (teoretycznie $-∞$) do zabronionych połączeń.

In [ ]:
def attention(q,k,v,mask=None):
    scores=q@k.transpose(-2,-1)/math.sqrt(q.size(-1))
    if mask is not None:scores=scores.masked_fill(~mask,torch.finfo(scores.dtype).min)
    weights=scores.softmax(-1)
    return weights@v,weights
B,T,dh=2,6,8
q=torch.randn(B,1,T,dh,device=device);k=torch.randn_like(q);v=torch.randn_like(q)
y,a=attention(q,k,v)
print("wyjście",y.shape,"wagi",a.shape,"sumy softmax",a.sum(-1)[0,0])

## 3. Causal masking
GPT przewiduje $x_t$ wyłącznie z $x_{<t}$. Maska dolnotrójkątna zezwala na klucze $j≤t$, blokując przyszłość. `True` oznacza dozwoloną pozycję.

In [ ]:
mask=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
plt.imshow(mask.cpu(),cmap="Greens");plt.xlabel("klucz j");plt.ylabel("zapytanie t");plt.title("Dozwolone połączenia");plt.show()
_,aw=attention(q,k,v,mask)
print("Maksymalna waga z przyszłości:",aw.masked_select(~mask).max().item())

## 4. Multi-head attention
Każda głowa ma własne projekcje i może reprezentować inną relację. Wyniki są konkatenowane i rzutowane: $MHA(X)=Concat(head_i)W^O$. Zwykle $d_k=d_{model}/h$.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout=0.):
        super().__init__(); assert d_model%n_heads==0
        self.h=n_heads;self.dh=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model);self.out=nn.Linear(d_model,d_model);self.drop=nn.Dropout(dropout)
    def forward(self,x,mask=None):
        B,T,C=x.shape;q,k,v=self.qkv(x).chunk(3,-1)
        def split(z):return z.view(B,T,self.h,self.dh).transpose(1,2)
        z,w=attention(split(q),split(k),split(v),mask)
        z=z.transpose(1,2).contiguous().view(B,T,C)
        return self.drop(self.out(z)),w
probe=torch.randn(2,7,32,device=device);mha=MultiHeadAttention(32,4).to(device)
y,w=mha(probe,torch.tril(torch.ones(7,7,dtype=torch.bool,device=device)))
print("wejście",probe.shape,"wyjście",y.shape,"mapy",w.shape)

## 5. Kodowanie pozycji
Self-attention bez informacji o kolejności jest równoważne względem permutacji tokenów. Kod sinusoidalny to $PE_{pos,2i}=sin(pos/10000^{2i/d})$ i $PE_{pos,2i+1}=cos(pos/10000^{2i/d})$. GPT często uczy embeddingi pozycji; nowsze warianty używają m.in. RoPE.

In [ ]:
class SinusoidalPosition(nn.Module):
    def __init__(self,d,max_len=256):
        super().__init__();pos=torch.arange(max_len).unsqueeze(1)
        div=torch.exp(torch.arange(0,d,2)*(-math.log(10000.)/d))
        pe=torch.zeros(max_len,d);pe[:,0::2]=torch.sin(pos*div);pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe",pe[None])
    def forward(self,x):return x+self.pe[:,:x.size(1)].to(x.dtype)
pos=SinusoidalPosition(32,64).to(device)
plt.figure(figsize=(8,3));plt.imshow(pos.pe[0].cpu().T,aspect="auto",cmap="coolwarm");plt.xlabel("pozycja");plt.ylabel("wymiar");plt.title("Kodowanie sinusoidalne");plt.show()

## 6. Blok decoder-only
Popularny wariant pre-LayerNorm ma residual attention i MLP: $X'=X+MHA(LN(X))$, $Y=X'+MLP(LN(X'))$. MLP zwykle rozszerza reprezentację do $4d$. GELU wnosi nieliniowość, residual ułatwia przepływ gradientu.

In [ ]:
class Block(nn.Module):
    def __init__(self,d,h,dropout=.1):
        super().__init__();self.ln1=nn.LayerNorm(d);self.attn=MultiHeadAttention(d,h,dropout);self.ln2=nn.LayerNorm(d)
        self.mlp=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d),nn.Dropout(dropout))
    def forward(self,x,mask):
        a,w=self.attn(self.ln1(x),mask);x=x+a;return x+self.mlp(self.ln2(x)),w
block=Block(32,4).to(device);out,maps=block(probe,torch.tril(torch.ones(7,7,dtype=torch.bool,device=device)))
print("wyjście",out.shape,"mapy",maps.shape,"parametry",sum(p.numel() for p in block.parameters()))

## 7. Koszt i ćwiczenia
Dla długości $T$ standardowe attention kosztuje $O(T²d)$ obliczeń i $O(T²)$ pamięci na głowę.

1. Zbuduj maskę lokalną, w której token widzi najwyżej 3 poprzednie pozycje.
2. Zmień liczbę głów przy stałym $d_{model}$ i sprawdź wymiary.
3. Zastąp sinusoidę uczonym `nn.Embedding(max_len,d_model)`.
4. Wyjaśnij, dlaczego permutacja wejścia bez pozycji permutuje wyniki.

## Podsumowanie
Q/K/V wyznaczają ważoną agregację wartości. Skala stabilizuje softmax, maska causal blokuje przyszłość, wiele głów łączy relacje, a kod pozycyjny wnosi kolejność. Blok łączy attention, MLP, LayerNorm i residual.